In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import odeint
import seaborn as sns

In [1]:

def sird_model(y, t, beta, gamma, mu):
    """
    SIRD model with death compartment

    TODO: Adapt the SIR model to include deaths

    Additional parameter:
    - mu: mortality rate

    New compartment:
    - D: Deaths
    """
    S, I, R, D = y  # Note: now 4 compartments.
    N = S + I + R + D

    # TODO: Modify the SIR equations to include death
    # Hint: Infectious people can either recover (rate γ) OR die (rate μ)

    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I - mu * I
    dRdt = gamma * I
    dDdt = mu * I

    return dSdt, dIdt, dRdt, dDdt

# Test your SIRD function
test_result = sird_model([990, 10, 0, 0], 0, 0.3, 0.1, 0.01)
print(f"SIRD test result: {test_result}")
print("You should see 4 numbers!")

SIRD test result: (-2.97, 1.87, 1.0, 0.1)
You should see 4 numbers!


In [ ]:
def run_gamma_sensitivity_analysis(gamma_values, beta=0.3, mu=0.01,
                                    S0=990, I0=10, R0_init=0, D0=0, days=160):
    """
    Run the SIRD model for each recovery rate in gamma_values and summarize
    the epidemic outcomes.

    Returns
    -------
    results_df : pandas.DataFrame
        Columns: ['gamma', 'R0', 'peak_infected', 'peak_day', 'total_deaths']
    fig : matplotlib.figure.Figure
        Epidemic curves (infectious individuals over time) for every gamma.
    """
    y0 = (S0, I0, R0_init, D0)
    t = np.linspace(0, days, days + 1)

    records = []
    curves = {}

    for gamma in gamma_values:
        solution = odeint(sird_model, y0, t, args=(beta, gamma, mu))
        S, I, R, D = solution.T

        peak_idx = np.argmax(I)
        records.append({
            'gamma': gamma,
            'R0': beta / gamma,
            'peak_infected': I[peak_idx],
            'peak_day': t[peak_idx],
            'total_deaths': D[-1],
        })
        curves[gamma] = I

    results_df = pd.DataFrame(records, columns=['gamma', 'R0', 'peak_infected', 'peak_day', 'total_deaths'])
    results_df = results_df.round({'gamma': 2, 'R0': 2, 'peak_infected': 1, 'peak_day': 0, 'total_deaths': 1})

    # --- Publication-quality plot ---
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(gamma_values)))

    for color, gamma in zip(colors, gamma_values):
        ax.plot(t, curves[gamma], color=color, linewidth=2.2,
                label=f'γ = {gamma} (R₀ = {beta / gamma:.2f})')

    ax.set_xlabel('Day', fontsize=12)
    ax.set_ylabel('Infectious individuals', fontsize=12)
    ax.set_title('SIRD Epidemic Curves for Varying Recovery Rates (γ)', fontsize=14, fontweight='bold')
    ax.legend(title='Recovery rate', frameon=False, fontsize=10)
    ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()

    return results_df, fig


# Run the sensitivity analysis across the required recovery rates
gamma_values = [0.05, 0.1, 0.15, 0.2, 0.25]
results_df, fig = run_gamma_sensitivity_analysis(gamma_values)

plt.show()
results_df